# Витрина транзакций ТЭ — 1. Факт операций

AI-14134. Одна строка = одна операция. Таблица `sandbox_ai.tmp_shestopalov_acq_trx`.

| Колонка | Источник `ods_alpha.scd1_trx` |
|---|---|
| `inn` | ИНН компании договора |
| `id_pos_terminal` | `c_nter` |
| `mcc_code` | `n_mcc`, MCC чека |
| `mcc_terminal` | MCC торговой точки, `scd1_merchants.n_mcc` по `c_nmrc` |
| `mcc_profile` | профиль точки, `scd1_merchants.c_mrc_profile` |
| `mcc_mismatch` | 1, если оба кода заполнены и не равны |
| `transaction_id` | `n_trx` |
| `date_t` | `d_trx_orig` |
| `sum_t` | `n_amt_src` |
| `status` | `cf_trx_stat` |
| `card_bin` | `n_bin` |
| `type_t` | `c_trx_type` |
| `payment_system` | по началу `n_bin`: 2200–2204 МИР, 4 Visa, 51–55 и 2221–2720 Mastercard, 62 UnionPay, остальное — прочие. Карты РСХБ тоже |
| `n_agr` | договор операции, `scd1_trx_acq` |
| `retl_id` | торговая точка, `c_nmrc` |
| `mrc_city` | город торговой точки, `c_mrc_city` |

Периметр договоров и отбор операций взяты из DAG витрины (`acq_01_sa_perimeter.sql`, `acq_05_trx_agr.sql`). Только покупки `S01`, как в DAG. В отличие от DAG статус не фильтруется: операции со статусом `R` остаются.

Окно — октябрь 2026. Даты стоят в SQL ниже: в `sa_agr` (два раза) и в `trx_base_raw`.

Порядок: `01` → `02` и `03` → `04`.


In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()


In [ ]:
with imp:
    imp.execute("set MEM_LIMIT=12g")
    imp.execute("drop table if exists sandbox_ai.tmp_shestopalov_acq_trx")
    imp.execute("""
create table sandbox_ai.tmp_shestopalov_acq_trx
as
with sa_agr as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
  from ods_alpha.scd1_agreements a
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  left semi join (
      select distinct cast(t.n_agr as string) as n_agr
      from ods_alpha.scd1_agr_terms t
      where upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and cast(t.d_valid_from as date) <= cast('2026-10-31' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('2026-10-01' as date))
    ) pt
    on pt.n_agr = cast(a.n_agr as string)
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
    and cast(a.d_valid_from as date) <= cast('2026-10-31' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('2026-10-01' as date))
),
fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
trx_base_raw as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(t.c_nter as string) as c_nter,
    t.n_mcc,
    t.d_trx_orig,
    t.n_amt_src,
    t.cf_trx_stat,
    cast(t.n_bin as string) as n_bin,
    t.c_trx_type,
    t.c_nmrc,
    t.c_mrc_city
  from ods_alpha.scd1_trx t
  join fiid_rshb fr
    on fr.c_fiid = cast(t.c_fiid_acq as string)
  where cast(t.d_trx_orig as timestamp) >= cast('2026-10-01' as timestamp)
    and cast(t.d_trx_orig as timestamp) < cast('2026-11-01' as timestamp)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
),
trx_base as (
  select
    n_trx,
    max(c_nter) as c_nter,
    max(n_mcc) as n_mcc,
    max(d_trx_orig) as d_trx_orig,
    max(n_amt_src) as n_amt_src,
    max(cf_trx_stat) as cf_trx_stat,
    max(n_bin) as n_bin,
    max(c_trx_type) as c_trx_type,
    max(c_nmrc) as c_nmrc,
    max(c_mrc_city) as c_mrc_city
  from trx_base_raw
  group by n_trx
),
ta as (
  select
    cast(a.n_trx as string) as n_trx,
    min(ss.n_agr) as n_agr
  from ods_alpha.scd1_trx_acq a
  join trx_base tb
    on tb.n_trx = cast(a.n_trx as string)
  join sa_agr ss
    on ss.n_agr = cast(a.n_agr as string)
  group by cast(a.n_trx as string)
),
outlet as (
  select c_nmrc, mcc_terminal, mcc_profile
  from (
    select
      cast(m.c_nmrc as string) as c_nmrc,
      cast(cast(m.n_mcc as bigint) as string) as mcc_terminal,
      nullif(trim(cast(m.c_mrc_profile as string)), '') as mcc_profile,
      row_number() over (
        partition by cast(m.c_nmrc as string)
        order by
          case when m.n_mcc is null then 1 else 0 end,
          case when nullif(trim(cast(m.c_mrc_profile as string)), '') is null then 1 else 0 end,
          cast(m.n_mcc as string)
      ) as rn
    from ods_alpha.scd1_merchants m
    left semi join trx_base tb
      on tb.c_nmrc = cast(m.c_nmrc as string)
    where coalesce(m.ods_deleted_flg, '0') <> '1'
      and m.c_nmrc is not null
  ) z
  where rn = 1
)
select
  ss.inn,
  tb.c_nter as id_pos_terminal,
  cast(cast(tb.n_mcc as bigint) as string) as mcc_code,
  o.mcc_terminal,
  o.mcc_profile,
  case
    when o.mcc_terminal is not null
     and tb.n_mcc is not null
     and o.mcc_terminal <> cast(cast(tb.n_mcc as bigint) as string)
    then 1
    else 0
  end as mcc_mismatch,
  tb.n_trx as transaction_id,
  tb.d_trx_orig as date_t,
  tb.n_amt_src as sum_t,
  tb.cf_trx_stat as status,
  tb.n_bin as card_bin,
  tb.c_trx_type as type_t,
  case
    when substring(tb.n_bin, 1, 4) between '2200' and '2204' then 'МИР'
    when substring(tb.n_bin, 1, 1) = '4' then 'Visa'
    when substring(tb.n_bin, 1, 2) between '51' and '55' then 'Mastercard'
    when substring(tb.n_bin, 1, 4) between '2221' and '2720' then 'Mastercard'
    when substring(tb.n_bin, 1, 2) = '62' then 'UnionPay'
    else 'прочие'
  end as payment_system,
  ta.n_agr,
  tb.c_nmrc as retl_id,
  tb.c_mrc_city as mrc_city
from trx_base tb
join ta
  on ta.n_trx = tb.n_trx
join sa_agr ss
  on ss.n_agr = ta.n_agr
left join outlet o
  on o.c_nmrc = cast(tb.c_nmrc as string)
""")
print("собрано sandbox_ai.tmp_shestopalov_acq_trx")

In [ ]:
with imp:
    display(imp.fetch("""
select
  count(*) as rows_cnt,
  count(distinct transaction_id) as trx_cnt,
  count(distinct inn) as inn_cnt,
  count(distinct n_agr) as agr_cnt,
  count(distinct id_pos_terminal) as terminal_cnt,
  min(date_t) as min_date,
  max(date_t) as max_date
from sandbox_ai.tmp_shestopalov_acq_trx
"""))
    display(imp.fetch("""
select type_t, status, count(*) as cnt
from sandbox_ai.tmp_shestopalov_acq_trx
group by type_t, status
order by cnt desc
"""))
    display(imp.fetch("""
select payment_system, length(card_bin) as bin_len, count(*) as cnt
from sandbox_ai.tmp_shestopalov_acq_trx
group by payment_system, length(card_bin)
order by cnt desc
"""))
    display(imp.fetch("""
select mrc_city, count(*) as cnt, count(distinct inn) as inn_cnt
from sandbox_ai.tmp_shestopalov_acq_trx
group by mrc_city
order by cnt desc
limit 30
"""))
    display(imp.fetch("select * from sandbox_ai.tmp_shestopalov_acq_trx limit 20"))
